In [ ]:
import numpy as np

# Single-qubit basis states
ket0 = np.array([1, 0], dtype=complex)   # |0⟩
ket1 = np.array([0, 1], dtype=complex)   # |1⟩

print("|0> =", ket0)
print("|1> =", ket1)

# Two-qubit basis states: |ab⟩ = |a⟩ ⊗ |b⟩ (q0 is the left digit)
ket00 = np.kron(ket0, ket0)
ket01 = np.kron(ket0, ket1)
ket10 = np.kron(ket1, ket0)
ket11 = np.kron(ket1, ket1)

print("|00> =", ket00)

In [ ]:
# Pauli-X: bit flip
X = np.array([[0, 1],
              [1, 0]], dtype=complex)

# Pauli-Z: phase flip
Z = np.array([[1,  0],
              [0, -1]], dtype=complex)

# Hadamard: creates superposition
H = np.array([[1,  1],
              [1, -1]], dtype=complex) / np.sqrt(2)

# Ry(θ): rotation by θ around the y axis
def Ry(theta):
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    return np.array([[c, -s],
                     [s,  c]], dtype=complex)

# Quick check: apply the gates to |0⟩
print("X|0>  =", X @ ket0)
print("Z|0>  =", Z @ ket0)
print("H|0>  =", H @ ket0)
print("Ry(pi)|0> =", np.round(Ry(np.pi) @ ket0, 3))

In [ ]:
I = np.eye(2, dtype=complex)   # identity matrix: do nothing

def apply_gate(state, gate, target):
    """Apply a single-qubit gate to a 2-qubit state.
    target = 0 -> q0 (left digit), target = 1 -> q1 (right digit)"""
    if target == 0:
        full = np.kron(gate, I)
    else:
        full = np.kron(I, gate)
    return full @ state

# Try 1: H on q0 of |00⟩
print(np.round(apply_gate(ket00, H, 0), 3))

# Try 2: X on q1 of |00⟩
print(np.round(apply_gate(ket00, X, 1), 3))

In [ ]:
# CNOT: control q0 (left), target q1 (right)
CNOT = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1],
                 [0, 0, 1, 0]], dtype=complex)

# Check all four basis states
for name, ket in [("00", ket00), ("01", ket01), ("10", ket10), ("11", ket11)]:
    print(name, "->", np.round(CNOT @ ket, 3).real)

In [ ]:
def measure(state, shots=1000):
    probs = np.abs(state) ** 2               # Born rule: |amplitude|²
    n = int(np.log2(len(state)))             # number of qubits (4 entries -> 2 qubits)
    labels = [format(i, f"0{n}b") for i in range(len(state))]   # "00","01","10","11"
    outcomes = np.random.choice(labels, size=shots, p=probs)     # weighted die
    counts = {l: int(np.sum(outcomes == l)) for l in labels}
    return counts

# Try: H on q0 of |00⟩ -> 1/√2 (|00⟩ + |10⟩)
state = apply_gate(ket00, H, 0)
print(measure(state))

In [ ]:
# Test 1: H·H|0⟩ = |0⟩
result = H @ (H @ ket0)        # inner H first, then outer H (right to left)
print("H·H|0> =", np.round(result, 3))

assert np.allclose(result, ket0), "Test 1 FAILED"
print("Test 1 passed ✅")

In [ ]:
# Test 2: Bell circuit
bell = ket00                        # start in |00⟩
bell = apply_gate(bell, H, 0)       # H on q0 -> 1/√2 (|00⟩ + |10⟩)
bell = CNOT @ bell                  # CNOT    -> 1/√2 (|00⟩ + |11⟩)
print("Bell state:", np.round(bell, 3).real)

counts = measure(bell, shots=1000)
print("Measurement:", counts)

assert counts["01"] == 0 and counts["10"] == 0, "Test 2 FAILED"
print("Test 2 passed ✅")